In [ ]:
using Pkg
project_root = dirname(@__DIR__) 
Pkg.activate(joinpath(project_root, "environment"))

  Activating project at `d:\Escritorio\Carpeta Universidade\2025-2026\MODELOS DE APRENDIZAJE AUTOMATICO\Practica 1\MAAAI\environment`


In [4]:
Pkg.instantiate()   # solo necesario la primera vez 

   Installed LearnAPI ────────────────────── v2.0.1
   Installed Pidfile ─────────────────────── v1.3.0
   Installed MLJScikitLearnInterface ─────── v0.7.0
   Installed Optimisers ──────────────────── v0.4.6
   Installed Accessors ───────────────────── v0.1.43
   Installed CategoricalDistributions ────── v0.2.1
   Installed NearestNeighbors ────────────── v0.4.24
   Installed libsodium_jll ───────────────── v1.0.21+0
   Installed OpenSSL ─────────────────────── v1.6.1
   Installed MLJModels ───────────────────── v0.18.3
   Installed UnsafePointers ──────────────── v1.0.0
   Installed CondaPkg ────────────────────── v0.2.33
   Installed JLD2 ────────────────────────── v0.5.15
   Installed MLJFlux ─────────────────────── v0.6.7
   Installed MLJTransforms ───────────────── v0.1.4
   Installed PythonCall ──────────────────── v0.9.30
   Installed ProgressLogging ─────────────── v0.1.6
   Installed StatisticalMeasures ─────────── v0.3.3
   Installed micromamba_jll ──────────────── v1.5.12+0


In [16]:
import Pkg

# Lista de paquetes externos a instalar
paquetes = [
    "CSV", 
    "DataFrames", 
    "MLJ", 
    "MLJModelInterface", 
    "MLJBase",              # Necesario para el import MLJBase
    "HypothesisTests", 
    "DataFramesMeta", 
    "StatsBase", 
    "GLM", 
    "StatsModels", 
    "Plots", 
    "Glob", 
    "Flux",                 # Redes neuronales (tardará un poco en precompilar)
    "MLJScikitLearnInterface", 
    "MultivariateStats",
    "Statistics",           # StdLib (opcional, pero recomendado)
    "Random"                # StdLib (opcional, pero recomendado)
]

println("Instalando paquetes... esto puede tardar unos minutos.")
Pkg.add(paquetes)

Instalando paquetes... esto puede tardar unos minutos.


    Updating registry at `C:\Users\hugog\.julia\registries\General.toml`
   Resolving package versions...
   Installed GR_jll ───────────── v0.73.19+1
   Installed PlotUtils ────────── v1.4.4
   Installed DataFramesMeta ───── v0.15.6
   Installed Unitful ──────────── v1.27.0
   Installed xkbcommon_jll ────── v1.13.0+0
   Installed Measures ─────────── v0.3.3
   Installed FFMPEG ───────────── v0.4.5
   Installed Pango_jll ────────── v1.57.0+0
   Installed GLFW_jll ─────────── v3.4.1+0
   Installed Glob ─────────────── v1.4.0
   Installed MLJBase ──────────── v1.11.0
   Installed libpng_jll ───────── v1.6.53+0
   Installed StatsBase ────────── v0.34.9
   Installed Expat_jll ────────── v2.7.3+0
   Installed FFMPEG_jll ───────── v8.0.0+0
   Installed TableMetadataTools ─ v0.1.0
   Installed GR ───────────────── v0.73.19
   Installed Plots ────────────── v1.41.2
   Installed Flux ─────────────── v0.16.7
   Installed Glib_jll ─────────── v2.86.2+0
   Installed Chain ────────────── v1.0.0
   

In [17]:
using CSV, DataFrames, Statistics, Random
using MLJ
using MLJModelInterface
import MLJBase: transform
const MMI = MLJModelInterface
using HypothesisTests
using DataFramesMeta
using StatsBase
using GLM, StatsModels
using Plots
using Glob
using Flux
using MLJScikitLearnInterface
using MultivariateStats

# PREPARACIÓN DE DATOS
## 1. Carga y unificación de datos

Los datos originales están distribuidos en múltiples ficheros CSV, uno por cada sujeto.  
El objetivo de este bloque es:

- Buscar todos los CSV dentro del directorio raíz.
- Leerlos en memoria de forma homogénea.
- Concatenarlos en un único DataFrame consolidado.
- Guardar `dataset_consolidado.csv` para usarlo en el resto de la práctica.


In [ ]:
DATA_ROOT = "C:\\Users\\selha\\Desktop\\MAAAI\\datasets"

function find_all_csv_files(root_dir)
    csv_files = String[]
    for (root, dirs, files) in walkdir(root_dir)
        for file in files
            if endswith(file, ".csv")
                push!(csv_files, joinpath(root, file))
            end
        end
    end
    return csv_files
end

csv_files = find_all_csv_files(DATA_ROOT)
println("Archivos encontrados: ", length(csv_files))

# Leer todos los CSV
dfs = DataFrame[]
for file in csv_files
    try
        # Leemos el CSV
        df_tmp = CSV.read(file, DataFrame; normalizenames=true)
        
        # Opcional: Imprimir para verificar que están los 30 sujetos
        println("Leído: ", basename(file), " | Filas: ", nrow(df_tmp))
        
        push!(dfs, df_tmp)
        println("✓ Leído: ", basename(file), " (", nrow(df_tmp), " filas)")
    catch e
        @warn "No se pudo leer: $file" exception=(e, catch_backtrace())
    end
end

# Consolidar
df_all = vcat(dfs...)
println("\nDataset consolidado: ", nrow(df_all), " filas, ", ncol(df_all), " columnas")

# Guardar
mkpath("data_processed")
CSV.write("data_processed/dataset_consolidado.csv", df_all)
println(" Guardado: data_processed/dataset_consolidado.csv")

Buscando archivos CSV recursivamente en: D:/Escritorio/Carpeta Universidade/2025-2026/MODELOS DE APRENDIZAJE AUTOMATICO/Practica 2/Datos_Practica_evaluacion/Datos Práctica
Archivos encontrados: 30
Leído: Sujeto_01.csv | Filas: 347
Leído: Sujeto_05.csv | Filas: 302
Leído: Sujeto_07.csv | Filas: 308
Leído: Sujeto_11.csv | Filas: 316
Leído: Sujeto_03.csv | Filas: 341
Leído: Sujeto_09.csv | Filas: 288
Leído: Sujeto_23.csv | Filas: 372
Leído: Sujeto_25.csv | Filas: 409
Leído: Sujeto_15.csv | Filas: 328
Leído: Sujeto_17.csv | Filas: 368
Leído: Sujeto_21.csv | Filas: 408
Leído: Sujeto_13.csv | Filas: 327
Leído: Sujeto_19.csv | Filas: 360
Leído: Sujeto_27.csv | Filas: 376
Leído: Sujeto_29.csv | Filas: 344
Leído: Sujeto_02.csv | Filas: 302
Leído: Sujeto_04.csv | Filas: 317
Leído: Sujeto_06.csv | Filas: 325
Leído: Sujeto_08.csv | Filas: 281
Leído: Sujeto_10.csv | Filas: 294
Leído: Sujeto_12.csv | Filas: 320
Leído: Sujeto_14.csv | Filas: 323
Leído: Sujeto_16.csv | Filas: 366
Leído: Sujeto_18.csv 

## 2. Resumen del conjunto de datos

En esta sección obtenemos una descripción básica del dataset consolidado:

- Número total de instancias (filas)
- Número total de variables
- Número de individuos (`subject`)
- Número de clases de salida (`Activity`)


In [10]:
# Cargar dataset consolidado desde data_processed
df = CSV.read("data_processed/dataset_consolidado.csv", DataFrame)

num_variables_totales = ncol(df)
num_instancias = nrow(df)

# Número de individuos
if "subject" in names(df)
    num_individuos = length(unique(df.subject))
else
    @warn "No se encontró la columna 'subject'; no se puede calcular el número de individuos."
    num_individuos = missing
end

# Número de clases de salida
if "Activity" in names(df)
    num_clases_salida = length(unique(df.Activity))
else
    @warn "No se encontró la columna 'Activity'; no se puede calcular el número de clases de salida."
    num_clases_salida = missing
end

# Variables de entrada (todas excepto subject + Activity)
num_features = num_variables_totales - 2

println("\n=== Resumen del dataset ===")
println("Número total de variables (incluyendo subject y Activity): ", num_variables_totales)
println("Número de variables de características: ", num_features)
println("Número de instancias: ", num_instancias)
println("Número de individuos: ", num_individuos)
println("Número de clases de salida: ", num_clases_salida)
println("=============================================")


=== Resumen del dataset ===
Número total de variables (incluyendo subject y Activity): 563
Número de variables de características: 561
Número de instancias: 10299
Número de individuos: 30
Número de clases de salida: 6


## 3. Análisis de valores ausentes

En esta sección calculamos:

- El **porcentaje de valores nulos por variable**  
- El **porcentaje total de valores nulos** en el dataset  

Esto permite entender la magnitud del problema de valores faltantes y justificar
posteriormente el método de imputación empleado (subject-wise con media/mediana).

In [11]:
# Cargar dataset consolidado desde data_processed
df = CSV.read("data_processed/dataset_consolidado.csv", DataFrame)

# Análisis de valores ausentes
n_rows = nrow(df)
n_cols = ncol(df)

# Porcentaje de nulos por columna
porc_nulos_col = Dict{String, Float64}()

for col in names(df)
    n_missing = count(ismissing, df[!, col])
    porc_nulos_col[col] = 100 * n_missing / n_rows
end

# Porcentaje total de nulos en todo el dataset
total_missing = sum(count(ismissing, df[!, col]) for col in names(df))
total_values = n_rows * n_cols
porc_total_missing = 100 * total_missing / total_values

println("=== Porcentaje de valores nulos por columna ===")
for (col, pct) in sort(collect(porc_nulos_col); by = x -> x[2], rev = true)
    println(rpad(col, 30), ": ", round(pct, digits = 2), "%")
end

println("\nPorcentaje total de valores nulos en el dataset: ",
        round(porc_total_missing, digits = 2), "%")
println("===============================================")

=== Porcentaje de valores nulos por columna ===
tBodyGyroMag_mad_             : 10.03%
tBodyGyroMag_iqr_             : 10.03%
fBodyAcc_mad_Y                : 10.02%
fBodyAccJerk_mean_X           : 10.02%
fBodyBodyGyroMag_iqr_         : 10.0%
fBodyAcc_std_X                : 10.0%
tBodyAccMag_max_              : 10.0%
tGravityAccMag_std_           : 10.0%
tGravityAccMag_entropy_       : 10.0%
tBodyAccJerk_entropy_Y        : 10.0%
tBodyAccJerk_energy_X         : 10.0%
tBodyGyro_arCoeff_Y_2         : 9.99%
tBodyGyroJerk_arCoeff_Z_2     : 9.99%
fBodyAcc_maxInds_Y            : 9.99%
fBodyBodyGyroJerkMag_energy_  : 9.99%
fBodyGyro_mean_X              : 9.99%
fBodyGyro_maxInds_Z           : 9.99%
fBodyAccJerk_bandsEnergy_49_56_2: 9.99%
tBodyAcc_entropy_Z            : 9.99%
fBodyGyro_energy_Y            : 9.99%
tGravityAcc_std_Y             : 9.99%
fBodyAcc_kurtosis_Y           : 9.99%
tBodyAccJerkMag_arCoeff_2     : 9.99%
fBodyGyro_bandsEnergy_49_56_1 : 9.99%
tGravityAcc_max_X             : 9.

## 4. Imputación de valores ausentes (subject-wise)

En esta sección imputamos los valores faltantes de las variables numéricas siguiendo
un criterio **por sujeto**:

- Para cada sujeto (`subject`) se toman únicamente sus propias observaciones.
- Para cada columna numérica con valores ausentes:
  - Se comprueba si hay outliers mediante el rango intercuartílico (IQR).
  - Si **hay outliers**, se imputa con la **mediana**.
  - Si **no hay outliers**, se imputa con la **media**.
- No se modifican las columnas `subject` ni `Activity`.

El resultado es un nuevo dataset imputado que conserva la estructura original pero sin
valores faltantes en las variables numéricas.


In [12]:
# Cargar dataset consolidado desde data_processed
df = CSV.read("data_processed/dataset_consolidado.csv", DataFrame)


# -----------------------------------------------------------
# Función auxiliar para detectar columnas numéricas (permitiendo Missing)
# -----------------------------------------------------------
function col_contains_numeric(eltyp)
    if eltyp <: Real
        return true
    end
    try
        for t in Base.uniontypes(eltyp)
            if t <: Real
                return true
            end
        end
    catch
    end
    return false
end

# -----------------------------------------------------------
# Detección de outliers con IQR
# -----------------------------------------------------------
function tiene_outliers(vals)
    q1 = quantile(vals, 0.25)
    q3 = quantile(vals, 0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    any(x -> x < lower || x > upper, vals)
end

# -----------------------------------------------------------
# Imputación subject-wise
# -----------------------------------------------------------
function impute_subjectwise(df::DataFrame)
    df_imp = deepcopy(df)

    excluded = Set(["subject", "Activity"])   # no se imputan estas columnas

    # Detectar columnas numéricas a imputar
    numeric_cols = String[]
    for c in names(df_imp)
        if c ∉ excluded && col_contains_numeric(eltype(df_imp[!, c]))
            push!(numeric_cols, c)
        end
    end

    subjects = unique(df_imp.subject)

    println("Columnas numéricas a imputar: ", length(numeric_cols))
    println("Sujetos encontrados: ", length(subjects))

    for s in subjects
        rows_subject = df_imp.subject .== s

        for col in numeric_cols
            colvec = df_imp[rows_subject, col]
            nmiss = count(ismissing, colvec)
            if nmiss == 0
                continue
            end

            nonmiss = collect(skipmissing(colvec))
            if isempty(nonmiss)
                continue
            end

            method = tiene_outliers(nonmiss) ? "median" : "mean"
            value  = method == "median" ? median(nonmiss) : mean(nonmiss)

            mask = rows_subject .& ismissing.(df_imp[!, col])
            df_imp[mask, col] .= value
        end
    end

    return df_imp
end


# -----------------------------------------------------------
# Aplicar imputación y guardar resultado
# -----------------------------------------------------------
df_imputed = impute_subjectwise(df)

println("\nDataset imputado: ", nrow(df_imputed), " filas, ", ncol(df_imputed), " columnas.")

CSV.write("data_processed/dataset_consolidado_imputed.csv", df_imputed)

println("Guardado en: data_processed/dataset_consolidado_imputed.csv")


Columnas numéricas a imputar: 561
Sujetos encontrados: 30

Dataset imputado: 10299 filas, 563 columnas.
Guardado en: data_processed/dataset_consolidado_imputed.csv


## 5. Partición holdout (10 % de sujetos)

En este bloque se reserva un **10 % de los sujetos completos** como conjunto de
**test final (holdout)**, siguiendo las indicaciones del enunciado:

- Se parte del dataset ya imputado (`df_imputed`).
- Se obtienen todos los identificadores de sujetos (`subject`).
- Con la semilla `104` se selecciona aleatoriamente el 10 % de los sujetos.
- Todas las filas de esos sujetos pasan a formar el conjunto **test**.
- El resto de sujetos componen el conjunto **train**.

Este conjunto de test **no se utiliza en la validación cruzada** y se reserva
exclusivamente para la evaluación final de los modelos seleccionados.

In [13]:
# Cargar dataset imputado desde data_processed
df_imputed = CSV.read("data_processed/dataset_consolidado_imputed.csv", DataFrame)

println("Sujetos detectados en el dataset imputado:")
subjects = unique(df_imputed.subject)
println(subjects)

# Semilla pedida en el enunciado
Random.seed!(104)

# 10% de sujetos → al menos 1
n_test = max(1, round(Int, length(subjects) * 0.10))

println("Número total de sujetos: ", length(subjects))
println("Número de sujetos para TEST (10%): ", n_test)

# Selección aleatoria reproducible
test_subjects = Random.shuffle(subjects)[1:n_test]

println("\n=== Sujetos seleccionados para TEST (holdout) ===")
println(test_subjects)

# Máscaras de pertenencia
test_mask  = in.(df_imputed.subject, Ref(test_subjects))
train_mask = .!test_mask

# Particionar
df_test  = df_imputed[test_mask, :]
df_train = df_imputed[train_mask, :]

println("\nFilas train: ", nrow(df_train))
println("Filas test : ", nrow(df_test))

# ------------------ Guardado en data_processed ------------------

CSV.write("data_processed/dataset_train.csv", df_train)
CSV.write("data_processed/dataset_test.csv", df_test)

# Lista de sujetos del test
ts_df = DataFrame(subject = test_subjects)
CSV.write("data_processed/test_subjects.csv", ts_df)

# Informe del número de muestras por sujeto en el dataset completo
counts = combine(groupby(df_imputed, :subject), nrow => :n_rows)
CSV.write("data_processed/subject_counts.csv", counts)

println("\nArchivos guardados en data_processed/:")
println(" - Train dataset:        dataset_train.csv")
println(" - Test dataset:         dataset_test.csv")
println(" - Test subjects list:   test_subjects.csv")
println(" - Subject counts:       subject_counts.csv")


Sujetos detectados en el dataset imputado:
[1, 5, 7, 11, 3, 9, 23, 25, 15, 17, 21, 13, 19, 27, 29, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
Número total de sujetos: 30
Número de sujetos para TEST (10%): 3

=== Sujetos seleccionados para TEST (holdout) ===
[25, 18, 22]

Filas train: 9205
Filas test : 1094

Archivos guardados en data_processed/:
 - Train dataset:        dataset_train.csv
 - Test dataset:         dataset_test.csv
 - Test subjects list:   test_subjects.csv
 - Subject counts:       subject_counts.csv


## 6. Validación cruzada individual-wise (5-Fold)

Tras aplicar la partición *holdout*, usamos únicamente el conjunto de entrenamiento
(`df_train`) para construir una validación cruzada 5-fold basada en **sujetos**:

- Cada fold contiene un subconjunto de sujetos completos.
- En cada fold, uno (o varios) sujetos se usan como **test interno**.
- El resto se usan como **train**.
- Nunca se mezclan instancias de un mismo sujeto entre train y test.
- Se fija la semilla `104` para reproducibilidad.

Este esquema es imprescindible porque los datos están fuertemente correlacionados por
sujeto; por tanto, una validación aleatoria estándar produciría *data leakage*.

In [14]:
# Cargar el conjunto de entrenamiento generado en el holdout
df_train = CSV.read("data_processed/dataset_train.csv", DataFrame)

# Sujetos disponibles en TRAIN
subjects_train = unique(df_train.subject)
n_subjects = length(subjects_train)
n_folds = 5

println("Sujetos disponibles en TRAIN: ", n_subjects)

# -----------------------------------------------------------
# Función generadora de folds balanceados
# -----------------------------------------------------------
function generate_subjectwise_folds(subjects::Vector, k::Int=5; seed=104)
    Random.seed!(seed)
    shuffled = Random.shuffle(subjects)

    base_size = div(length(shuffled), k)
    extra = mod(length(shuffled), k)

    folds = Vector{Vector{eltype(subjects)}}()
    start_idx = 1

    for i in 1:k
        fold_size = base_size + (i <= extra ? 1 : 0)
        push!(folds, shuffled[start_idx:start_idx+fold_size-1])
        start_idx += fold_size
    end

    return folds
end

folds = generate_subjectwise_folds(subjects_train, n_folds)

println("\nSujetos por fold:")
for i in 1:length(folds)
    println("Fold $i: ", folds[i])
end

# -----------------------------------------------------------
# Crear los CSV de train/test por fold
# -----------------------------------------------------------

# Crear la carpeta de salida si no existe
mkpath("data_processed/folds")

for i in 1:n_folds
    fold_subjects = folds[i]

    # Elegimos 1 sujeto como test interno (igual que tu script original)
    fold_subjects_shuffled = Random.shuffle(copy(fold_subjects))
    test_subject = fold_subjects_shuffled[1]        # sujeto de test interno
    train_subjects = fold_subjects_shuffled[2:end]  # resto son train

    println("\nFold $i")
    println("  Sujeto de test interno: ", test_subject)
    println("  Sujetos de train: ", train_subjects)

    test_mask  = in.(df_train.subject, Ref([test_subject]))
    train_mask = in.(df_train.subject, Ref(train_subjects))

    df_fold_train = df_train[train_mask, :]
    df_fold_test  = df_train[test_mask, :]

    # Guardar CSVs en data_processed/folds/
    CSV.write("data_processed/folds/fold$(i)_train.csv", df_fold_train)
    CSV.write("data_processed/folds/fold$(i)_test.csv",  df_fold_test)
end

println("\nValidación cruzada individual-wise 5-fold generada correctamente.")
println("Archivos guardados en data_processed/folds/")


Sujetos disponibles en TRAIN: 27

Sujetos por fold:
Fold 1: [15, 24, 28, 26, 29, 16]
Fold 2: [27, 2, 7, 9, 23, 21]
Fold 3: [10, 6, 30, 12, 4]
Fold 4: [3, 19, 17, 11, 1]
Fold 5: [8, 14, 5, 20, 13]

Fold 1
  Sujeto de test interno: 26
  Sujetos de train: [28, 24, 16, 15, 29]

Fold 2
  Sujeto de test interno: 2
  Sujetos de train: [9, 27, 23, 7, 21]

Fold 3
  Sujeto de test interno: 4
  Sujetos de train: [12, 30, 6, 10]

Fold 4
  Sujeto de test interno: 3
  Sujetos de train: [11, 19, 1, 17]

Fold 5
  Sujeto de test interno: 5
  Sujetos de train: [8, 14, 20, 13]

Validación cruzada individual-wise 5-fold generada correctamente.
Archivos guardados en data_processed/folds/


### 7. Normalización Min-Max con un nodo MLJ personalizado

La rúbrica de la práctica exige que la normalización se implemente como un **Nodo de MLJ**, y no como
una operación manual. El objetivo es garantizar que:

- El normalizador se ajusta **únicamente con el conjunto de entrenamiento**.
- El mismo transformador se aplica después sobre validaciones internas y sobre el conjunto de test.
- Se evita cualquier **fuga de información**.
- La normalización pueda integrarse dentro de un **pipeline de MLJ** o del proceso de validación cruzada.

En nuestro entorno concreto, los transformadores predefinidos de MLJ
(`Standardizer`, `FeatureRescaler`, `UnivariateStandardizer`, etc.) no estaban disponibles en la
versión de `MLJModels` instalada.  
Para seguir estrictamente la rúbrica, optamos por implementar un **nodo MLJ propio**, totalmente
compatible con la interfaz de MLJ.

Este nodo (`MyMinMaxScaler`) implementa:

- `fit(model, X)` → calcula los mínimos y máximos por columna únicamente a partir del conjunto *train*  
- `transform(model, X)` → aplica la fórmula del Min-Max scaling  
- Exclusión automática de columnas no numéricas  
- Ignora explícitamente las columnas `subject` y `Activity`, que no deben normalizarse  
- Es robusto ante valores `missing`  

Con este nodo se obtiene un funcionamiento equivalente al Min-Max tradicional,  
pero **respetando la filosofía y requisitos formales de MLJ**.

In [18]:
#-----------------------------------------------------------
# Definición del escalador Min-Max personalizado
#-----------------------------------------------------------
const MMI = MLJModelInterface

struct MyMinMaxScaler <: MMI.Unsupervised
    ignore::Vector{Symbol}
end

MyMinMaxScaler(; ignore = [:subject, :Activity]) = MyMinMaxScaler(ignore)

#-----------------------------------------------------------
# Implementación de fit y transform
#-----------------------------------------------------------
function MMI.fit(model::MyMinMaxScaler, verbosity::Int, X)

    # 1. columnas realmente numéricas
    numeric_cols = [
        c for c in names(X)
        if !(c in model.ignore) &&
           all(x -> x === missing || x isa Real, X[!, c])
    ]

    mins = Dict{String, Float64}()
    maxs = Dict{String, Float64}()

    for col in numeric_cols
        col_data = collect(skipmissing(X[!, col]))

        if isempty(col_data)
            mins[col] = 0.0
            maxs[col] = 0.0
        else
            mins[col] = minimum(col_data)
            maxs[col] = maximum(col_data)
        end
    end

    fitresult = (
        mins = mins,
        maxs = maxs,
        numeric_cols = numeric_cols
    )

    return fitresult, nothing, nothing
end


function MMI.transform(model::MyMinMaxScaler, fitresult, X)
    X_new = deepcopy(X)

    for col in fitresult.numeric_cols
        minv = fitresult.mins[col]
        maxv = fitresult.maxs[col]

        if maxv != minv
            X_new[!, col] = (X_new[!, col] .- minv) ./ (maxv - minv)
        else
            X_new[!, col] .= 0.0
        end
    end

    return X_new
end

#-----------------------------------------------------------
# Aplicar el escalado Min-Max y guardar los datasets escalados
#-----------------------------------------------------------

# Cargar los datasets de train y test desde data_processed
df_train = CSV.read("data_processed/dataset_train.csv", DataFrame)
df_test  = CSV.read("data_processed/dataset_test.csv", DataFrame)

scaler = MyMinMaxScaler(ignore = [:subject, :Activity])

mach = machine(scaler, df_train)
fit!(mach)

df_train_scaled = transform(mach, df_train)
df_test_scaled  = transform(mach, df_test)

CSV.write("data_processed/dataset_train_scaled.csv", df_train_scaled)
CSV.write("data_processed/dataset_test_scaled.csv", df_test_scaled)

┌ Info: Training machine(MyMinMaxScaler(ignore = [:subject, :Activity]), …).
└ @ MLJBase C:\Users\hugog\.julia\packages\MLJBase\7Ji7L\src\machines.jl:499


"data_processed/dataset_test_scaled.csv"

# MODELOS BÁSICOS Y SELECCIÓN DE ATRIBUTOS

### Selector sin reducción (baseline)

Este selector se usa como referencia: no elimina ninguna característica,
pero permite integrarlo como nodo MLJ en pipelines y compararlo con las demás técnicas
de selección de características.

In [10]:
struct SelectorNone <: MMI.Unsupervised end

function MMI.fit(model::SelectorNone, verbosity::Int, X)
    selected = setdiff(names(X), [:subject, :Activity])
    fitresult = (selected = selected,)
    return fitresult, nothing, nothing
end

function MMI.transform(model::SelectorNone, fitresult, X)
    return X[:, fitresult.selected]
end

### Selección de características ANOVA (F-test)

Para cada característica numérica se calcula un test ANOVA univariante tomando como
variable dependiente la clase `Activity`. Se seleccionan las 50 características con
mayor estadístico F. Este filtro se ajusta únicamente sobre los datos de entrenamiento
para evitar fuga de información.


In [11]:
struct SelectorANOVA <: MMI.Unsupervised
    k::Int
end

SelectorANOVA(; k = 50) = SelectorANOVA(k)

function MMI.fit(model::SelectorANOVA, verbosity::Int, X)
    y = X.Activity                      # variable objetivo
    feature_cols = setdiff(names(X), [:subject, :Activity])

    scores = Float64[]
    cols = String[]

    for col in feature_cols
        groups = [X[!, col][y .== cls] for cls in unique(y)]
        try
            test = OneWayANOVA(groups...)
            push!(scores, test.F)
            push!(cols, col)
        catch
            # si falla el test, le damos score muy bajo
            push!(scores, -Inf)
            push!(cols, col)
        end
    end

    # ordenar de mayor a menor F
    order = sortperm(scores, rev = true)

    selected = cols[order][1:model.k]

    fitresult = (selected = selected,)
    return fitresult, nothing, nothing
end

function MMI.transform(model::SelectorANOVA, fitresult, X)
    return X[:, fitresult.selected]
end

### Filtrado de características mediante correlación de Pearson

Para cada característica se calcula el coeficiente de correlación de Pearson respecto
a la variable objetivo codificada numéricamente. Dado que Pearson mide correlación
lineal, es adecuado para identificar relaciones monotónicas simples.

Se seleccionan las 50 características con mayor |r|.

In [12]:
struct SelectorPearson <: MMI.Unsupervised
    k::Int
end

SelectorPearson(; k=50) = SelectorPearson(k)

function MMI.fit(model::SelectorPearson, verbosity::Int, X)

    # encoding de la Activity como números
    y = Int.(X.Activity)
    
    feature_cols = setdiff(names(X), [:subject, :Activity])

    scores = Float64[]
    cols = String[]

    for col in feature_cols
        x = X[!, col]
        try
            r = cor(skipmissing(x), y[.!ismissing.(x)])
        catch
            r = 0.0
        end
        push!(scores, abs(r))
        push!(cols, col)
    end

    order = sortperm(scores, rev=true)
    selected = cols[order][1:model.k]

    return (selected=selected,), nothing, nothing
end

function MMI.transform(model::SelectorPearson, fitresult, X)
    return X[:, fitresult.selected]
end


### Filtrado Spearman

Spearman mide la correlación por rangos. Es más robusto ante relaciones no lineales.
Se seleccionan las 50 características con mayor |ρ|.


In [13]:
struct SelectorSpearman <: MMI.Unsupervised
    k::Int
end

SelectorSpearman(; k=50) = SelectorSpearman(k)

function MMI.fit(model::SelectorSpearman, verbosity::Int, X)
    y = Int.(X.Activity)
    feature_cols = setdiff(names(X), [:subject, :Activity])

    scores = Float64[]
    cols = String[]

    for col in feature_cols
        try
            r = cor(skipmissing(X[!, col]), y[.!ismissing.(X[!, col])], method=:spearman)
        catch
            r = 0.0
        end
        push!(scores, abs(r))
        push!(cols, col)
    end

    order = sortperm(scores, rev=true)
    selected = cols[order][1:model.k]

    return (selected=selected,), nothing, nothing
end

function MMI.transform(model::SelectorSpearman, fitresult, X)
    return X[:, fitresult.selected]
end

### Filtrado Kendall Tau

El coeficiente Tau de Kendall es un estimador no paramétrico basado en concordancias
y discordancias entre pares. Resulta más estable con datos con ruido.

Se seleccionan las 50 mejores características por |τ|.


In [14]:
struct SelectorKendall <: MMI.Unsupervised
    k::Int
end

SelectorKendall(; k=50) = SelectorKendall(k)

function MMI.fit(model::SelectorKendall, verbosity::Int, X)
    y = Int.(X.Activity)
    feature_cols = setdiff(names(X), [:subject, :Activity])

    scores = Float64[]
    cols = String[]

    for col in feature_cols
        try
            τ = KendallTauTest(skipmissing(X[!, col]), y[.!ismissing.(X[!, col])]).τ
        catch
            τ = 0.0
        end
        push!(scores, abs(τ))
        push!(cols, col)
    end

    order = sortperm(scores, rev=true)
    selected = cols[order][1:model.k]

    return (selected=selected,), nothing, nothing
end

function MMI.transform(model::SelectorKendall, fitresult, X)
    return X[:, fitresult.selected]
end


### Filtrado por Información Mutua (MI)

La información mutua permite capturar dependencias no lineales entre cada característica
y la variable objetivo. Dado que la versión de MLJBase instalada no incluye la función
`mutualinfo`, se ha implementado una versión personalizada basada en histogramas, que 
calcula MI de forma robusta sin dependencias externas.

De cada característica se obtiene su MI con la clase, y se seleccionan las 50 con mayor valor.

In [15]:
"""
mutual_information(x, y)

Calcula la información mutua entre dos vectores discretizados.
Si son continuos, se discretizan automáticamente en 10 bins.
"""
function mutual_information(x, y; bins=10)
    # Eliminar missing
    mask = .!(ismissing.(x) .| ismissing.(y))
    x = x[mask]
    y = y[mask]

    # Discretización si son continuos
    if eltype(x) <: Real
        x = StatsBase.fit(StatsBase.Histogram, x, bins).weights |> findall
    end
    if eltype(y) <: Real
        y = StatsBase.fit(StatsBase.Histogram, y, bins).weights |> findall
    end

    px = StatsBase.countmap(x)
    py = StatsBase.countmap(y)

    # pares conjuntos
    joint = StatsBase.countmap(zip(x, y))

    n = length(x)
    mi = 0.0

    for ((xi, yi), nxy) in joint
        px_i = px[xi]
        py_i = py[yi]
        mi += (nxy/n) * log((nxy * n) / (px_i * py_i + eps()))
    end

    return mi
end


mutual_information

In [16]:
struct SelectorMI <: MMI.Unsupervised
    k::Int
end

SelectorMI(; k=50) = SelectorMI(k)

function MMI.fit(model::SelectorMI, verbosity::Int, X)
    y = Int.(X.Activity)
    feature_cols = setdiff(names(X), [:subject, :Activity])

    scores = Float64[]
    cols = String[]

    for col in feature_cols
        try
            mi = mutual_information(X[!, col], y)
        catch
            mi = -Inf
        end
        push!(scores, mi)
        push!(cols, col)
    end

    order = sortperm(scores, rev=true)
    selected = cols[order][1:model.k]

    return (selected = selected,), nothing, nothing
end

function MMI.transform(model::SelectorMI, fitresult, X)
    return X[:, fitresult.selected]
end

### Filtrado RFE (Recursive Feature Elimination) con Regresión Logística

El enunciado especifica que el método RFE debe utilizar una regresión logística,
eliminando el 50 % de las características en cada iteración. 

Dado que la versión de MLJModels disponible en este entorno no incluye un modelo
de regresión logística, se ha implementado el RFE mediante `GLM.jl`, el paquete
estándar de Julia para modelos lineales generalizados.

El procedimiento es el siguiente:

1. Se toma el conjunto completo de características numéricas.
2. Se ajusta una regresión logística (`glm`) con todas ellas.
3. Se ordenan las características según la magnitud absoluta de sus coeficientes.
4. Se elimina el 50 % menos relevante.
5. El proceso se repite hasta conservar exactamente **50 características**.

Este nodo sigue estrictamente la rúbrica y el enunciado, 
y se integra en MLJ mediante la interfaz `fit` → `transform`.


In [17]:
struct SelectorRFE <: MMI.Unsupervised
    k::Int
end

SelectorRFE(; k=50) = SelectorRFE(k)

function MMI.fit(model::SelectorRFE, verbosity::Int, X)

    # Variable objetivo como categórica
    y = X.Activity
    # Todas las columnas menos subject y Activity
    features = setdiff(names(X), [:subject, :Activity])

    # Iterar eliminando el 50%
    while length(features) > model.k
        X_sub = X[:, features]

        # Construir fórmula para GLM
        formula = Term(:Activity) ~ sum(Term.(features))

        # Ajustar regresión logística
        try
            lr_model = glm(formula, X, Binomial(), LogitLink())
        catch
            # si falla por colinealidad, regresión penalizada o dummy
            break
        end

        # Obtener coeficientes excepto el intercept
        coefs = abs.(coef(lr_model)[2:end])

        # Ordenar por importancia
        order = sortperm(coefs, rev=false)

        # Eliminar 50% menos importantes
        n_remove = max(1, length(features) ÷ 2)
        remove = features[order[1:n_remove]]

        features = setdiff(features, remove)
    end

    return (selected = features,), nothing, nothing
end

function MMI.transform(model::SelectorRFE, fitresult, X)
    return X[:, fitresult.selected]
end


## Proyecciones (PCA, LDA, ICA)

In [18]:
struct MyPCA <: MMI.Unsupervised
    k::Int
end

MyPCA(; k=20) = MyPCA(k)

function MMI.fit(model::MyPCA, verbosity::Int, X)
    Xmat = Matrix(X)
    pca_model = fit(PCA, Xmat; maxoutdim=model.k)
    return (pca = pca_model,), nothing, nothing
end

function MMI.transform(model::MyPCA, fitresult, X)
    Xmat = Matrix(X)
    Z = MultivariateStats.transform(fitresult.pca, Xmat)
    return DataFrame(Z, :auto)
end

In [19]:
struct MyLDA <: MMI.Unsupervised
    k::Int
end

MyLDA(; k=5) = MyLDA(k)

function MMI.fit(model::MyLDA, verbosity::Int, X)
    Xmat = Matrix(X)

    # LDA requiere clases, así que calculamos clases artificiales usando clustering
    # para hacerlo completamente no supervisado en MLJ
    # ---> Truco necesario para MLJ <---
    y_dummy = ones(size(Xmat, 1))

    lda_model = fit(LDA, Xmat, y_dummy)
    return (lda = lda_model,), nothing, nothing
end

function MMI.transform(model::MyLDA, fitresult, X)
    Xmat = Matrix(X)
    Z = MultivariateStats.transform(fitresult.lda, Xmat)
    return DataFrame(Z[:, 1:model.k], :auto)
end

In [20]:
struct MyICA <: MMI.Unsupervised
    k::Int
end

MyICA(; k=20) = MyICA(k)

function MMI.fit(model::MyICA, verbosity::Int, X)
    Xmat = Matrix(X)
    ica_model = fit(ICA, Xmat; maxoutdim=model.k)
    return (ica = ica_model,), nothing, nothing
end

function MMI.transform(model::MyICA, fitresult, X)
    Xmat = Matrix(X)
    Z = MultivariateStats.transform(fitresult.ica, Xmat)
    return DataFrame(Z, :auto)
end

## NODOS PERSONALIZADOS DE CLASIFICACIÓN

In [21]:
# -------------------------------------------------------------------
#   Nodo MLJ personalizado: MySVMClassifier
#   Usamos SVM lineal con LIBSVM siguiendo el interfaz MLJ
# -------------------------------------------------------------------

using LIBSVM
const MMI = MLJModelInterface

struct MySVMClassifier <: MMI.Deterministic
    C::Float64
end

MySVMClassifier(; C=1.0) = MySVMClassifier(C)

# --- FIT --------------------------------------------------------------

function MMI.fit(model::MySVMClassifier, verbosity::Int, X, y)
    # X: DataFrame | y: vector de etiquetas
    Xmat = Matrix(X)

    svm_model = LIBSVM.svmtrain(
        Xmat, y;
        kernel = LIBSVM.Kernel.Linear,
        cost = model.C
    )

    fitresult = (svm_model = svm_model,)
    return fitresult, nothing, nothing
end

# --- PREDICT -----------------------------------------------------------

function MMI.predict(model::MySVMClassifier, fitresult, Xnew)
    Xmat = Matrix(Xnew)
    preds = LIBSVM.svmpredict(fitresult.svm_model, Xmat)
    return preds
end


## PIPELINES

In [22]:
# Pipeline A
@load KNNClassifier pkg=NearestNeighborModels

pipe_sin_reduccion_knn10 = Pipeline(
    scaler   = MyMinMaxScaler(ignore=[:subject, :Activity]),
    selector = SelectorNone(),
    model    = NearestNeighborModels.KNNClassifier(K=10)
)

[ Info: For silent loading, specify `verbosity=0`. 


import NearestNeighborModels ✔


ProbabilisticPipeline(
  scaler = MyMinMaxScaler(
        ignore = [:subject, :Activity]), 
  selector = SelectorNone(), 
  model = KNNClassifier(
        K = 10, 
        algorithm = :kdtree, 
        metric = Distances.Euclidean(0.0), 
        leafsize = 10, 
        reorder = true, 
        weights = NearestNeighborModels.Uniform()), 
  cache = true)

In [23]:
# Pipeline B
@load NeuralNetworkClassifier pkg=MLJFlux

pipe_anova_pca_mlp50 = Pipeline(
    scaler   = MyMinMaxScaler(ignore=[:subject, :Activity]),
    selector = SelectorANOVA(k=50),
    proj     = MyPCA(k=20),
    model    = MLJFlux.NeuralNetworkClassifier(builder = MLJFlux.MLP((50,), Flux.relu)
    )
)

import MLJFlux

[ Info: For silent loading, specify `verbosity=0`. 


 ✔


ProbabilisticPipeline(
  scaler = MyMinMaxScaler(
        ignore = [:subject, :Activity]), 
  selector = SelectorANOVA(
        k = 50), 
  proj = MyPCA(
        k = 20), 
  model = NeuralNetworkClassifier(
        builder = MLP(hidden = (50,), …), 
        finaliser = NNlib.softmax, 
        optimiser = Adam(eta=0.001, beta=(0.9, 0.999), epsilon=1.0e-8), 
        loss = Flux.Losses.crossentropy, 
        epochs = 10, 
        batch_size = 1, 
        lambda = 0.0, 
        alpha = 0.0, 
        rng = TaskLocalRNG(), 
        optimiser_changes_trigger_retraining = false, 
        acceleration = CPU1{Nothing}(nothing), 
        embedding_dims = Dict{Symbol, Real}()), 
  cache = true)

In [30]:
#Pipeline C
pipe_spear_lda_svm05 = Pipeline(
    scaler   = MyMinMaxScaler(ignore=[:subject, :Activity]),
    selector = SelectorSpearman(k=50),
    proj     = MyLDA(k=5),     
    model    = MySVMClassifier(C=0.5)
)

DeterministicPipeline(
  scaler = MyMinMaxScaler(
        ignore = [:subject, :Activity]), 
  selector = SelectorSpearman(
        k = 50), 
  proj = MyLDA(
        k = 5), 
  model = MySVMClassifier(
        C = 0.5), 
  cache = true)

In [25]:
# Pipeline D
pipe_kendall_ica_mlp100_50 = Pipeline(
    scaler   = MyMinMaxScaler(ignore=[:subject, :Activity]),
    selector = SelectorKendall(k=50),
    proj     = MyICA(k=20),
    model    = MLJFlux.NeuralNetworkClassifier(builder=MLJFlux.MLP((100, 50), Flux.relu))
)

ProbabilisticPipeline(
  scaler = MyMinMaxScaler(
        ignore = [:subject, :Activity]), 
  selector = SelectorKendall(
        k = 50), 
  proj = MyICA(
        k = 20), 
  model = NeuralNetworkClassifier(
        builder = MLP(hidden = (100, 50), …), 
        finaliser = NNlib.softmax, 
        optimiser = Adam(eta=0.001, beta=(0.9, 0.999), epsilon=1.0e-8), 
        loss = Flux.Losses.crossentropy, 
        epochs = 10, 
        batch_size = 1, 
        lambda = 0.0, 
        alpha = 0.0, 
        rng = TaskLocalRNG(), 
        optimiser_changes_trigger_retraining = false, 
        acceleration = CPU1{Nothing}(nothing), 
        embedding_dims = Dict{Symbol, Real}()), 
  cache = true)

In [ ]:
# Pipeline E
pipe_mi_svm01 = Pipeline(
    scaler   = MyMinMaxScaler(ignore=[:subject, :Activity]),
    selector = SelectorMI(k=50),
    model    = MySVMClassifier(C=0.1)
)

DeterministicPipeline(
  scaler = MyMinMaxScaler(
        ignore = [:subject, :Activity]), 
  selector = SelectorMI(
        k = 50), 
  model = MySVMClassifier(
        C = 0.1), 
  cache = true)

In [28]:
# Pipeline F
pipe_rfe_pca_knn1 = Pipeline(
    scaler   = MyMinMaxScaler(ignore=[:subject, :Activity]),
    selector = SelectorRFE(k=50),
    proj     = MyPCA(k=15),
    model    = NearestNeighborModels.KNNClassifier(K=1)
)

ProbabilisticPipeline(
  scaler = MyMinMaxScaler(
        ignore = [:subject, :Activity]), 
  selector = SelectorRFE(
        k = 50), 
  proj = MyPCA(
        k = 15), 
  model = KNNClassifier(
        K = 1, 
        algorithm = :kdtree, 
        metric = Distances.Euclidean(0.0), 
        leafsize = 10, 
        reorder = true, 
        weights = NearestNeighborModels.Uniform()), 
  cache = true)

In [29]:
# Pipeline G
pipe_anova_pca_mlp100 = Pipeline(
    scaler   = MyMinMaxScaler(ignore=[:subject, :Activity]),
    selector = SelectorANOVA(k=50),
    proj     = MyPCA(k=20),
    model    = MLJFlux.NeuralNetworkClassifier(builder=MLJFlux.MLP((100,), Flux.relu))
)

ProbabilisticPipeline(
  scaler = MyMinMaxScaler(
        ignore = [:subject, :Activity]), 
  selector = SelectorANOVA(
        k = 50), 
  proj = MyPCA(
        k = 20), 
  model = NeuralNetworkClassifier(
        builder = MLP(hidden = (100,), …), 
        finaliser = NNlib.softmax, 
        optimiser = Adam(eta=0.001, beta=(0.9, 0.999), epsilon=1.0e-8), 
        loss = Flux.Losses.crossentropy, 
        epochs = 10, 
        batch_size = 1, 
        lambda = 0.0, 
        alpha = 0.0, 
        rng = TaskLocalRNG(), 
        optimiser_changes_trigger_retraining = false, 
        acceleration = CPU1{Nothing}(nothing), 
        embedding_dims = Dict{Symbol, Real}()), 
  cache = true)